Data is in the "content" folder.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
import pandas as pd
import requests
import time
import glob
import yfinance as yf
from google.colab import drive
from pathlib import Path

## 1. Data Collection


### 1.1 Raw Data Collection


*   Source: Yahoo Finance, accessed programmatically via the yfinance Python library.
*   Stocks: NVIDIA (NVDA), AMD (AMD), Intel (INTC), Qualcomm (QCOM), Broadcom (AVGO), Texas Instruments (TXN), Micron (MU), Marvell (MRVL), Analog Devices (ADI), Microchip Technology (MCHP), ON Semiconductor (ON), and Monolithic Power Systems (MPWR)

*   Data Range: 2016-06-30 to 2026-06-30 (inclusive)
*   Columns: Date, Adjust Close, Close, Dividends, High, Low, Open, Stock Splits, Volume






In [4]:
def collect_raw_stock_data(tickers, start, end, output_dir):
    """Download daily OHLCV data for each ticker from Yahoo Finance,
    add the daily return, and save one CSV per ticker.

    Note: yfinance treats `end` as exclusive, so pass the day AFTER
    the last date you want included.
    """
    os.makedirs(output_dir, exist_ok=True)

    for ticker in tickers:
        print(f"Downloading {ticker}...")

        df = yf.download(
            ticker,
            start=start,
            end=end,
            interval="1d",
            auto_adjust=False,
            actions=True,
        )

        # Flatten the (field, ticker) MultiIndex columns so the CSV
        # gets a normal single-row header
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.droplevel(1)

        df.reset_index(inplace=True)

        # Daily return from Adj Close so dividends and splits
        # don't show up as fake price moves
        df["Daily_Return"] = df["Adj Close"].pct_change()

        csv_path = os.path.join(output_dir, f"stock_data_{ticker}_10y.csv")
        df.to_csv(csv_path, index=False)

        print(f"Saved: {csv_path} ({len(df)} rows)")

    print("\nDone!")


# Semiconductor stocks
tickers = [
    "NVDA", "AMD", "INTC", "QCOM", "AVGO", "TXN",
    "MU", "MRVL", "ADI", "MCHP", "ON", "MPWR"
]

drive.mount('/content/drive')
output_dir = '/content/drive/MyDrive/raw_data'

collect_raw_stock_data(
    tickers,
    start="2016-07-01",
    end="2026-07-01",   # exclusive, so data runs through 2026-06-30
    output_dir=output_dir,
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


[*********************100%***********************]  1 of 1 completed


Saved: /content/drive/MyDrive/raw_data/stock_data_NVDA_10y.csv (2512 rows)


[*********************100%***********************]  1 of 1 completed


Saved: /content/drive/MyDrive/raw_data/stock_data_AMD_10y.csv (2512 rows)


...

Saved: /content/drive/MyDrive/raw_data/stock_data_MCHP_10y.csv (2512 rows)


[*********************100%***********************]  1 of 1 completed


Saved: /content/drive/MyDrive/raw_data/stock_data_ON_10y.csv (2512 rows)


[*********************100%***********************]  1 of 1 completed

Saved: /content/drive/MyDrive/raw_data/stock_data_MPWR_10y.csv (2512 rows)

Done!


### 1.1 Raw Stock Data Collection - Daily Return


*   Compute daily returns and write them back into the raw stock data CSVs.
*   Uses Adj Close so dividends and splits don't show up as fake price moves.



In [5]:
def add_daily_return(raw_dir, pattern="stock_data_*_10y.csv"):
    """Compute the daily return from Adj Close and write it back
    into each raw stock data CSV in `raw_dir`.

    Adj Close is used so dividends and splits don't show up as
    fake price moves. Handles files fresh from yf.download that
    still have the second header row holding the ticker, so it is
    safe to run on old files and safe to re-run.
    """
    csv_files = sorted(glob.glob(os.path.join(raw_dir, pattern)))

    if not csv_files:
        print(f"No files matching {pattern} in {raw_dir}")
        return

    for csv_path in csv_files:
        # Detect the ticker header row (MultiIndex columns from
        # yf.download): its first cell is empty
        with open(csv_path) as f:
            f.readline()
            has_ticker_row = f.readline().startswith(",")

        df = pd.read_csv(
            csv_path,
            skiprows=[1] if has_ticker_row else None,
            parse_dates=["Date"],
        )

        df["Daily_Return"] = df["Adj Close"].pct_change()

        df.to_csv(csv_path, index=False)

        print(f"Updated {os.path.basename(csv_path)}: {len(df)} rows")

    print("\nDone!")


raw_dir = '/content/drive/MyDrive/raw_data'
add_daily_return(raw_dir)

Updated stock_data_ADI_10y.csv: 2512 rows
Updated stock_data_AMD_10y.csv: 2512 rows
Updated stock_data_AVGO_10y.csv: 2512 rows
Updated stock_data_INTC_10y.csv: 2512 rows
Updated stock_data_MCHP_10y.csv: 2512 rows
Updated stock_data_MPWR_10y.csv: 2512 rows
Updated stock_data_MRVL_10y.csv: 2512 rows
Updated stock_data_MU_10y.csv: 2512 rows
Updated stock_data_NVDA_10y.csv: 2512 rows
Updated stock_data_ON_10y.csv: 2512 rows
Updated stock_data_QCOM_10y.csv: 2512 rows
Updated stock_data_TXN_10y.csv: 2512 rows

Done!


## 1.2 Data Collection Company Quarterly Financial Information


*   Collect company fundamentals from SEC XBRL company facts (10-Q / 10-K)
*   revenue, gross profit (+ margin), R&D expense, CapEx, EPS (basic/diluted,
split-adjusted), net income, operating / investing / financing cash flow,
net change in cash, inventory, debt and stockholders' equity (+ D/E)


*   Cash-flow statements in 10-Qs are year-to-date only, so quarterly cash
flows are derived by differencing consecutive year-to-date spans (Q2 = 6-month YTD - Q1, Q3 = 9-month - 6-month, Q4 = annual - 9-month).Rows containing any such derived value are flagged `has_derived_flows`.






In [6]:
HEADERS = {"User-Agent": "CS6140 class project zhang.yanan4@northeastern.edu"}

# Period-end window: starts before the stock window so trailing
# fundamentals exist at 2016-06-30
START_DATE = "2015-01-01"
END_DATE = "2026-06-30"

# MRVL and AVGO changed legal entities mid-window, so both the current
# and the predecessor CIK are needed to cover 2016-2026
TICKER_TO_CIKS = {
    "NVDA": ["0001045810"],
    "AMD": ["0000002488"],
    "INTC": ["0000050863"],
    "QCOM": ["0000804328"],
    "AVGO": ["0001730168", "0001649338"],  # Broadcom Inc + Broadcom Ltd
    "TXN": ["0000097476"],
    "MU": ["0000723125"],
    "MRVL": ["0001835632", "0001058057"],  # Marvell Inc + Marvell Group Ltd
    "ADI": ["0000006281"],
    "MCHP": ["0000827054"],
    "ON": ["0001097864"],
    "MPWR": ["0001280452"],
}

# Stock splits inside the window (from the raw price data). Reported EPS
# is divided by the product of ratios of splits AFTER the filing date,
# putting every EPS value on the current post-split share basis.
SPLITS = {
    "NVDA": [("2021-07-20", 4.0), ("2024-06-10", 10.0)],
    "AVGO": [("2024-07-15", 10.0)],
    "MCHP": [("2021-10-13", 2.0)],
}

# kind "flow" facts cover a start..end period (income / cash-flow
# statement); "instant" facts are balances on a single date (balance
# sheet). Tags are in priority order - first match wins on ties.
METRICS = {
    "revenue_usd": dict(kind="flow", unit="USD", tags=[
        "RevenueFromContractWithCustomerExcludingAssessedTax",
        "Revenues",
        "SalesRevenueNet",
        "RevenueFromContractWithCustomerIncludingAssessedTax",
    ]),
    "cost_of_revenue_usd": dict(kind="flow", unit="USD", tags=[
        "CostOfRevenue",
        "CostOfGoodsAndServicesSold",
        "CostOfGoodsSold",
    ]),
    "gross_profit_usd": dict(kind="flow", unit="USD", tags=["GrossProfit"]),
    "rd_expense_usd": dict(kind="flow", unit="USD", tags=[
        "ResearchAndDevelopmentExpense",
        "ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost",
    ]),
    "capex_usd": dict(kind="flow", unit="USD", tags=[
        "PaymentsToAcquirePropertyPlantAndEquipment",
        "PaymentsToAcquireProductiveAssets",
        "CapitalExpenditures",
    ]),
    "eps_basic": dict(kind="flow", unit="USD/shares", tags=["EarningsPerShareBasic"]),
    "eps_diluted": dict(kind="flow", unit="USD/shares", tags=["EarningsPerShareDiluted"]),
    "net_income_usd": dict(kind="flow", unit="USD", tags=[
        "NetIncomeLoss",
        "ProfitLoss",
    ]),
    "operating_cf_usd": dict(kind="flow", unit="USD", tags=[
        "NetCashProvidedByUsedInOperatingActivities",
        "NetCashProvidedByUsedInOperatingActivitiesContinuingOperations",
    ]),
    "investing_cf_usd": dict(kind="flow", unit="USD", tags=[
        "NetCashProvidedByUsedInInvestingActivities",
        "NetCashProvidedByUsedInInvestingActivitiesContinuingOperations",
    ]),
    "financing_cf_usd": dict(kind="flow", unit="USD", tags=[
        "NetCashProvidedByUsedInFinancingActivities",
        "NetCashProvidedByUsedInFinancingActivitiesContinuingOperations",
    ]),
    "net_change_in_cash_usd": dict(kind="flow", unit="USD", tags=[
        "CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalentsPeriodIncreaseDecreaseIncludingExchangeRateEffect",
        "CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalentsPeriodIncreaseDecreaseExcludingExchangeRateEffect",
        "CashAndCashEquivalentsPeriodIncreaseDecrease",
    ]),
    "inventory_usd": dict(kind="instant", unit="USD", tags=["InventoryNet"]),
    "lt_debt_noncurrent_usd": dict(kind="instant", unit="USD", tags=[
        "LongTermDebtNoncurrent",
        "LongTermDebtAndCapitalLeaseObligations",
        "LongTermDebt",
    ]),
    "lt_debt_current_usd": dict(kind="instant", unit="USD", tags=[
        "LongTermDebtCurrent",
        "LongTermDebtAndCapitalLeaseObligationsCurrent",
        "DebtCurrent",
        "ShortTermBorrowings",
    ]),
    "stockholders_equity_usd": dict(kind="instant", unit="USD", tags=[
        "StockholdersEquity",
        "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest",
    ]),
}

# Flow metrics that can be differenced across year-to-date spans and
# derived as fiscal year minus Q1+Q2+Q3. EPS included as an
# approximation - the average share count drifts slightly within a year.
ADDITIVE = [
    "revenue_usd", "cost_of_revenue_usd", "gross_profit_usd",
    "rd_expense_usd", "capex_usd", "net_income_usd",
    "operating_cf_usd", "investing_cf_usd", "financing_cf_usd",
    "net_change_in_cash_usd", "eps_basic", "eps_diluted",
]

# duration in days -> reporting span
SPAN_BOUNDS = [
    (60, 120, "quarterly"),
    (150, 210, "ytd2"),      # 6-month year-to-date
    (240, 300, "ytd3"),      # 9-month year-to-date
    (330, 400, "annual"),
]


def split_factor(ticker, filed):
    """Product of split ratios announced after `filed`."""
    factor = 1.0
    for date, ratio in SPLITS.get(ticker, []):
        if filed < date:
            factor *= ratio
    return factor


def fetch_company_facts(cik, cache_dir=None):
    if cache_dir:
        cache = Path(cache_dir) / f"CIK{cik}.json"
        if cache.exists():
            return json.loads(cache.read_text())
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    response = requests.get(url, headers=HEADERS, timeout=60)
    response.raise_for_status()
    data = response.json()
    if cache_dir:
        Path(cache_dir).mkdir(parents=True, exist_ok=True)
        cache.write_text(json.dumps(data))
    time.sleep(0.2)
    return data


def extract_metric_rows(us_gaap, metric, spec, ticker):
    """All usable facts for one metric from one SEC entity."""
    rows = []
    for priority, tag in enumerate(spec["tags"]):
        for item in us_gaap.get(tag, {}).get("units", {}).get(spec["unit"], []):
            if item.get("form") not in ("10-K", "10-Q"):
                continue
            end, filed, val = item.get("end"), item.get("filed"), item.get("val")
            if end is None or filed is None or val is None:
                continue
            if spec["kind"] == "flow":
                start = item.get("start")
                if start is None:
                    continue
                days = (pd.Timestamp(end) - pd.Timestamp(start)).days
                freq = next((name for lo, hi, name in SPAN_BOUNDS
                             if lo <= days <= hi), None)
                if freq is None:
                    continue
                if freq in ("ytd2", "ytd3") and metric not in ADDITIVE:
                    continue
            else:
                start, freq = "", "instant"
            if metric.startswith("eps_"):
                val = val / split_factor(ticker, filed)
            rows.append({"metric": metric, "start": start, "end": end,
                         "freq": freq, "val": val, "filed": filed,
                         "priority": priority})
    return rows


def collect_one_ticker(ticker, ciks, cache_dir=None):
    rows = []
    for cik in ciks:
        facts = fetch_company_facts(cik, cache_dir)
        us_gaap = facts.get("facts", {}).get("us-gaap", {})
        for metric, spec in METRICS.items():
            rows += extract_metric_rows(us_gaap, metric, spec, ticker)

    if not rows:
        return pd.DataFrame()

    long = pd.DataFrame(rows)
    long["filed"] = pd.to_datetime(long["filed"])

    # One value per (metric, period): keep the EARLIEST filing so numbers
    # are as originally reported - later filings repeat prior periods as
    # comparatives, and using them would leak restated values back in time
    long = (long.sort_values(["filed", "priority"])
                .groupby(["metric", "start", "end"], as_index=False)
                .first())
    long = long[(long["end"] >= START_DATE) & (long["end"] <= END_DATE)]

    direct_q = long[long["freq"] == "quarterly"]
    annual = long[long["freq"] == "annual"]
    instant = long[long["freq"] == "instant"]

    # Quarterly flows from year-to-date differences: two spans sharing a
    # start date and ending one quarter apart give the quarter between
    # their end dates (6mo - 3mo = Q2, 9mo - 6mo = Q3, FY - 9mo = Q4)
    derived = []
    ytd = long[long["freq"] != "instant"]
    ytd = ytd[ytd["metric"].isin(ADDITIVE)]
    for (metric, start), grp in ytd.groupby(["metric", "start"]):
        grp = grp.sort_values("end")
        prev = None
        for _, cur in grp.iterrows():
            if prev is not None:
                gap = (pd.Timestamp(cur["end"]) - pd.Timestamp(prev["end"])).days
                if 60 <= gap <= 120:
                    derived.append({"metric": metric, "end": cur["end"],
                                    "val": cur["val"] - prev["val"],
                                    "filed": max(cur["filed"], prev["filed"])})
            prev = cur

    # Directly reported quarterly values win over derived ones
    q_long = direct_q[["metric", "end", "val", "filed"]].assign(derived=False)
    if derived:
        q_long = pd.concat(
            [q_long, pd.DataFrame(derived).assign(derived=True)],
            ignore_index=True)
    q_long = (q_long.sort_values("derived")
                    .groupby(["metric", "end"], as_index=False).first())

    def to_wide(sub):
        if sub.empty:
            return pd.DataFrame()
        wide = sub.pivot(index="end", columns="metric", values="val")
        wide["filed_date"] = (sub.pivot(index="end", columns="metric",
                                        values="filed").max(axis=1))
        return wide

    quarterly = to_wide(q_long)
    if not quarterly.empty and "derived" in q_long.columns:
        flags = q_long.pivot(index="end", columns="metric", values="derived")
        quarterly["has_derived_flows"] = flags.fillna(False).any(axis=1)
    annual_w = to_wide(annual[["metric", "end", "val", "filed"]])
    instant_w = to_wide(instant[["metric", "end", "val", "filed"]])

    # Fallback Q4 derivation (annual minus the three quarters inside the
    # fiscal year) for anything the year-to-date chain didn't cover
    for fy_end, fy_row in annual_w.iterrows():
        lo = (pd.Timestamp(fy_end) - pd.Timedelta(days=350)).strftime("%Y-%m-%d")
        window = quarterly[(quarterly.index > lo) & (quarterly.index < fy_end)]
        if len(window) != 3:
            continue
        if fy_end not in quarterly.index:
            quarterly.loc[fy_end, "filed_date"] = fy_row["filed_date"]
            quarterly.loc[fy_end, "has_derived_flows"] = False
        for metric in ADDITIVE:
            if metric not in annual_w.columns or metric not in quarterly.columns:
                continue
            if pd.isna(fy_row[metric]) or window[metric].isna().any():
                continue
            if pd.notna(quarterly.loc[fy_end, metric]):
                continue
            quarterly.loc[fy_end, metric] = fy_row[metric] - window[metric].sum()
            quarterly.loc[fy_end, "has_derived_flows"] = True
    quarterly = quarterly.sort_index()

    # Attach balance-sheet snapshots to matching period ends
    def attach_instant(flows, how):
        inst = instant_w.rename(columns={"filed_date": "filed_inst"})
        merged = flows.join(inst, how=how)
        merged["filed_date"] = merged[["filed_date", "filed_inst"]].max(axis=1)
        return merged.drop(columns=["filed_inst"])

    quarterly = attach_instant(quarterly, how="outer")
    annual_w = attach_instant(annual_w, how="left")

    def finalize(df, freq):
        df = df.copy()
        for metric in METRICS:
            if metric not in df.columns:
                df[metric] = pd.NA
            df[metric] = pd.to_numeric(df[metric], errors="coerce")
        # Gross profit: derive from revenue - cost of revenue where not
        # tagged directly (QCOM never tags GrossProfit)
        df["gross_profit_usd"] = df["gross_profit_usd"].fillna(
            df["revenue_usd"] - df["cost_of_revenue_usd"])
        df["gross_margin"] = df["gross_profit_usd"] / df["revenue_usd"]
        # Missing debt on a reported balance sheet means no debt
        # (e.g. MPWR carries none), so treat NaN components as 0
        df["total_debt_usd"] = (df["lt_debt_noncurrent_usd"].fillna(0)
                                + df["lt_debt_current_usd"].fillna(0))
        df["debt_to_equity"] = df["total_debt_usd"] / df["stockholders_equity_usd"]
        df.loc[df["stockholders_equity_usd"].isna(), "total_debt_usd"] = pd.NA
        df["ticker"] = ticker
        df["freq"] = freq
        if "has_derived_flows" not in df.columns:
            df["has_derived_flows"] = False
        df["has_derived_flows"] = (df["has_derived_flows"].astype("boolean")
                                   .fillna(False).astype(bool))
        df = df.reset_index().rename(columns={"end": "period_end",
                                              "index": "period_end"})
        cols = (["ticker", "period_end", "freq", "filed_date",
                 "has_derived_flows"]
                + list(METRICS)
                + ["gross_margin", "total_debt_usd", "debt_to_equity"])
        return df[cols]

    return pd.concat([finalize(quarterly, "quarterly"),
                      finalize(annual_w, "annual")],
                     ignore_index=True)


def collect_sec_fundamentals(out_dir, cache_dir=None):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    all_data = []
    for ticker, ciks in TICKER_TO_CIKS.items():
        print(f"Collecting {ticker} from SEC...")
        try:
            df = collect_one_ticker(ticker, ciks, cache_dir)
            if df.empty:
                print(f"  No data for {ticker}")
                continue
            df.to_csv(out_dir / f"financial_info_{ticker}_10y.csv", index=False)
            all_data.append(df)
            n_q = (df["freq"] == "quarterly").sum()
            n_a = (df["freq"] == "annual").sum()
            print(f"  {n_q} quarterly + {n_a} annual periods, "
                  f"{df['period_end'].min()}..{df['period_end'].max()}")
        except Exception as e:
            print(f"  Error collecting {ticker}: {e}")

    combined = pd.concat(all_data, ignore_index=True)
    combined.to_csv(out_dir / "all_semiconductor_sec_fundamentals.csv", index=False)
    combined[combined["freq"] == "quarterly"].to_csv(
        out_dir / "all_semiconductor_sec_quarterly.csv", index=False)
    combined[combined["freq"] == "annual"].to_csv(
        out_dir / "all_semiconductor_sec_annual.csv", index=False)

    print(f"\nDone! Combined shape: {combined.shape}")
    return combined


fundamentals = collect_sec_fundamentals(out_dir="data/fundamentals_sec")
fundamentals.head()

/tmp/ipykernel_653/3731741853.py:239: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  quarterly["has_derived_flows"] = flags.fillna(False).any(axis=1)


  46 quarterly + 12 annual periods, 2015-01-25..2026-04-26
  45 quarterly + 11 annual periods, 2015-03-28..2026-03-28
  45 quarterly + 11 annual periods, 2015-03-28..2026-03-28


/tmp/ipykernel_653/3731741853.py:239: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  quarterly["has_derived_flows"] = flags.fillna(False).any(axis=1)


  45 quarterly + 11 annual periods, 2015-03-29..2026-03-29
  46 quarterly + 11 annual periods, 2015-02-01..2026-05-03
  45 quarterly + 11 annual periods, 2015-03-31..2026-03-31


/tmp/ipykernel_653/3731741853.py:239: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  quarterly["has_derived_flows"] = flags.fillna(False).any(axis=1)


...

  46 quarterly + 12 annual periods, 2015-01-31..2026-05-02
  46 quarterly + 11 annual periods, 2015-01-31..2026-05-02


/tmp/ipykernel_653/3731741853.py:239: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  quarterly["has_derived_flows"] = flags.fillna(False).any(axis=1)


  46 quarterly + 12 annual periods, 2015-03-31..2026-03-31
  45 quarterly + 11 annual periods, 2015-04-03..2026-04-03
  46 quarterly + 11 annual periods, 2015-03-31..2026-03-31

Done! Combined shape: (683, 24)


/tmp/ipykernel_653/3731741853.py:239: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  quarterly["has_derived_flows"] = flags.fillna(False).any(axis=1)


metric,ticker,period_end,freq,filed_date,has_derived_flows,revenue_usd,cost_of_revenue_usd,gross_profit_usd,rd_expense_usd,capex_usd,...,investing_cf_usd,financing_cf_usd,net_change_in_cash_usd,inventory_usd,lt_debt_noncurrent_usd,lt_debt_current_usd,stockholders_equity_usd,gross_margin,total_debt_usd,debt_to_equity
0,NVDA,2015-01-25,quarterly,2015-03-12,False,1.250514e+09,550911000.0,699603000.0,NaN,NaN,...,NaN,NaN,NaN,482893000.0,1.384342e+09,NaN,4.417982e+09,0.559452,1.384342e+09,0.313343
1,NVDA,2015-04-26,quarterly,2015-05-20,False,1.151000e+09,498000000.0,653000000.0,339000000.0,30000000.0,...,-235000000.0,-44000000.0,-33000000.0,438000000.0,1.391000e+09,NaN,4.556000e+09,0.567333,1.391000e+09,0.305312
2,NVDA,2015-07-26,quarterly,2015-08-19,True,1.153000e+09,519000000.0,634000000.0,320000000.0,24000000.0,...,247000000.0,-439000000.0,-29000000.0,441000000.0,1.399000e+09,NaN,4.185000e+09,0.549870,1.399000e+09,0.334289
3,NVDA,2015-10-25,quarterly,2015-11-18,True,1.305000e+09,571000000.0,734000000.0,329000000.0,17000000.0,...,-199000000.0,-20000000.0,36000000.0,425000000.0,1.406000e+09,NaN,4.465000e+09,0.562452,1.406000e+09,0.314894
4,NVDA,2016-01-31,quarterly,2016-03-17,True,1.401000e+09,610000000.0,791000000.0,344000000.0,NaN,...,-213000000.0,-173000000.0,125000000.0,418000000.0,1.413000e+09,NaN,4.469000e+09,0.564597,1.413000e+09,0.316178


In [7]:

# Insider trading information
# Only gives us the last 7 transactions unfortunately -DC
import os
import yfinance as yf

tickers = [
    "NVDA", "AMD", "INTC", "QCOM", "AVGO", "TXN",
    "MU", "MRVL", "ADI", "MCHP", "ON", "MPWR"
]

output_dir = "insider_purchases"

os.makedirs(output_dir, exist_ok=True)

for ticker in tickers:
    print(f"Downloading insider purchases for {ticker}...")

    stock = yf.Ticker(ticker)

    try:
        insider_purchases = stock.insider_purchases

        if insider_purchases is not None and not insider_purchases.empty:
            csv_path = os.path.join(
                output_dir,
                f"{ticker}_insider_purchases.csv"
            )

            insider_purchases.to_csv(csv_path, index=True)

            print(f"Saved: {csv_path}")
        else:
            print(f"No insider purchase data found for {ticker}")

    except Exception as e:
        print(f"Error downloading {ticker}: {e}")

print("\nDone!")

Saved: insider_purchases/NVDA_insider_purchases.csv
Saved: insider_purchases/AMD_insider_purchases.csv
Saved: insider_purchases/INTC_insider_purchases.csv
Saved: insider_purchases/QCOM_insider_purchases.csv
Saved: insider_purchases/AVGO_insider_purchases.csv
Saved: insider_purchases/TXN_insider_purchases.csv
Saved: insider_purchases/MU_insider_purchases.csv
Saved: insider_purchases/MRVL_insider_purchases.csv
Saved: insider_purchases/ADI_insider_purchases.csv
Saved: insider_purchases/MCHP_insider_purchases.csv
Saved: insider_purchases/ON_insider_purchases.csv
Saved: insider_purchases/MPWR_insider_purchases.csv

Done!


In [8]:
# Testing technical indicators -DC
import pandas as pd
import numpy as np

def add_technical_indicators(df):
    df = df.copy()

    # Daily returns
    df["Daily_Return"] = df["Close"].pct_change()
    df["Log_Return"] = np.log(df["Close"] / df["Close"].shift(1))

    # Moving averages
    df["SMA_5"] = df["Close"].rolling(window=5).mean()
    df["SMA_10"] = df["Close"].rolling(window=10).mean()
    df["SMA_20"] = df["Close"].rolling(window=20).mean()
    df["SMA_50"] = df["Close"].rolling(window=50).mean()

    # Exponential moving averages
    df["EMA_12"] = df["Close"].ewm(span=12, adjust=False).mean()
    df["EMA_26"] = df["Close"].ewm(span=26, adjust=False).mean()

    # MACD
    df["MACD"] = df["EMA_12"] - df["EMA_26"]
    df["MACD_Signal"] = df["MACD"].ewm(span=9, adjust=False).mean()
    df["MACD_Hist"] = df["MACD"] - df["MACD_Signal"]

    # RSI
    delta = df["Close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(window=14).mean()
    avg_loss = loss.rolling(window=14).mean()

    rs = avg_gain / avg_loss
    df["RSI_14"] = 100 - (100 / (1 + rs))

    # Bollinger Bands
    df["BB_Middle"] = df["Close"].rolling(window=20).mean()
    bb_std = df["Close"].rolling(window=20).std()

    df["BB_Upper"] = df["BB_Middle"] + (2 * bb_std)
    df["BB_Lower"] = df["BB_Middle"] - (2 * bb_std)
    df["BB_Width"] = df["BB_Upper"] - df["BB_Lower"]

    # Volatility
    df["Volatility_10"] = df["Daily_Return"].rolling(window=10).std()
    df["Volatility_20"] = df["Daily_Return"].rolling(window=20).std()

    # Volume indicators
    df["Volume_SMA_20"] = df["Volume"].rolling(window=20).mean()
    df["Relative_Volume"] = df["Volume"] / df["Volume_SMA_20"]

    # Lag features
    df["Return_Lag_1"] = df["Daily_Return"].shift(1)
    df["Return_Lag_2"] = df["Daily_Return"].shift(2)
    df["Return_Lag_3"] = df["Daily_Return"].shift(3)


    # ML
    # 20-Day Price Momentum (Percentage change over 20 days)
    df["Momentum_20"] = df["Close"].pct_change(periods=20)

    # Distance from 20-Day MA ((Current - MA) / MA)
    df["Dist_SMA_20"] = (df["Close"] - df["SMA_20"]) / df["SMA_20"]

    # 5-Day Return (Percentage change over 5 days)
    df["Return_5d"] = df["Close"].pct_change(periods=5)

    # ATR (Average True Range - typically uses a 14-day period)
    high_low = df["High"] - df["Low"]
    high_close = (df["High"] - df["Close"].shift(1)).abs()
    low_close = (df["Low"] - df["Close"].shift(1)).abs()

    # True Range is the maximum of the three
    true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)

    # Average True Range (14-day SMA of True Range)
    df["ATR_14"] = true_range.rolling(window=14).mean()


    # Target variable: will tomorrow close be higher?
    df["Target"] = (df["Close"].shift(-1) > df["Close"]).astype(int)

    # Drop rows with NaN values from rolling calculations
    df.dropna(inplace=True)

    return df

In [9]:
# Run processing of raw data to get technical data
# saved in filepath /content/technical_data -DC
import glob
import os
import pandas as pd

input_dir = "/content/raw_data"
output_dir = "/content/technical_data"

os.makedirs(output_dir, exist_ok=True)

csv_files = glob.glob(os.path.join(input_dir, "*.csv"))

for csv_file in csv_files:
    filename = os.path.basename(csv_file)

    print(f"Processing {filename}...")

    df = pd.read_csv(csv_file)

    # Remove the bad ticker row
    df = df[df["Date"].notna()]

    # Convert Date
    df["Date"] = pd.to_datetime(df["Date"])

    # Convert numeric columns
    numeric_cols = [
        "Open", "High", "Low", "Close", "Adj Close",
        "Volume", "Dividends", "Stock Splits"
    ]

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Add technical indicators
    df = add_technical_indicators(df)

    # Save the processed file
    output_path = os.path.join(output_dir, filename)
    df.to_csv(output_path, index=False)

    print(f"Saved: {output_path}")

print("\nDone!")


Done!


In [10]:
# ML
def add_lag_features(df):
    """
    Adds lagged return features to an individual stock's dataframe.
    """
    df_lag = df.copy()

    # Calculate rolling returns, then shift by 1 day to ensure
    # the model only sees data available prior to the prediction date.
    df_lag["Prev_1d_Return"] = df_lag["Close"].pct_change(periods=1).shift(1)
    df_lag["Prev_5d_Return"] = df_lag["Close"].pct_change(periods=5).shift(1)
    df_lag["Prev_20d_Return"] = df_lag["Close"].pct_change(periods=20).shift(1)
    df_lag["Prev_60d_Return"] = df_lag["Close"].pct_change(periods=60).shift(1)

    return df_lag

